# Exploración de datos de Instacart con Python y Pandas

## Momento 1: Comprensión del problema

**1. Descripción del problema:**
Como analista de datos de Instacart, mi objetivo es explorar 5 conjuntos de datos transaccionales para comprender el comportamiento de los consumidores y reconocer patrones de compra, asegurando la calidad de la información antes del análisis profundo.

**2. Función de los 5 archivos:**
- `instacart_orders.csv`: Contiene la cabecera de las facturas (quién compra, cuándo y a qué hora).
- `products.csv`: Catálogo estático con los nombres de todos los productos y sus clasificaciones.
- `aisles.csv`: Catálogo de los pasillos (categorías específicas).
- `departments.csv`: Catálogo de departamentos (macro-categorías).
- `order_products.csv`: El detalle de cada factura (qué productos exactos llevó cada cliente en cada pedido).

**3. Columnas identificadoras (Llaves):**
- **Pedidos:** `order_id`
- **Usuarios:** `user_id`
- **Productos:** `product_id`
- **Categorías:** `aisle_id` y `department_id`

**4. Preguntas de análisis:**
1. ¿A qué hora del día se realiza la mayor cantidad de pedidos?
2. ¿Qué proporción de productos comprados suelen ser recompras (reordered)?
3. ¿Cuántos días en promedio tardan los clientes en volver a realizar un pedido?

**5. Importancia de explorar los datos:**
Es vital explorar los datos antes de analizarlos para identificar valores nulos, validar que los tipos de datos sean correctos (números vs. texto) y conocer el tamaño de las tablas. Esto previene errores de cálculo y caídas de memoria en fases posteriores.

In [1]:
import pandas as pd

# Importación de los archivos (con separador de punto y coma)
orders = pd.read_csv("instacart_orders.csv", sep=";")
products = pd.read_csv("products.csv", sep=";")
aisles = pd.read_csv("aisles.csv", sep=";")
departments = pd.read_csv("departments.csv", sep=";")
order_products = pd.read_csv("order_products.csv", sep=";")

# Observamos las dimensiones (filas, columnas) de cada archivo
print("Dimensiones de Orders:", orders.shape)
print("Dimensiones de Products:", products.shape)
print("Dimensiones de Aisles:", aisles.shape)
print("Dimensiones de Departments:", departments.shape)
print("Dimensiones de Order_products:", order_products.shape)

Dimensiones de Orders: (478967, 6)
Dimensiones de Products: (49694, 4)
Dimensiones de Aisles: (134, 2)
Dimensiones de Departments: (21, 2)
Dimensiones de Order_products: (4545007, 4)


## Respuestas del Momento 2:

- **¿Cuál de los cinco DataFrames contiene más registros?** 
  El DataFrame `order_products` es el más grande, con 4.545.007 registros.
  
- **¿Cuál contiene menos columnas?** 
  Los DataFrames `aisles` y `departments` contienen la menor cantidad, con solo 2 columnas cada uno.
  
- **¿Qué diferencias identifica entre los datos de pedidos y los datos de productos?** 
  Los datos de pedidos (`orders`) registran eventos dinámicos en el tiempo (transacciones, horarios, días), mientras que los datos de productos (`products`) son un catálogo estático de inventario que no cambia con cada compra.
  
- **¿Por qué es importante conocer las dimensiones de un conjunto de datos?** 
  Para dimensionar el consumo de memoria del computador, entender la granularidad de la información y confirmar que los archivos se importaron completos sin perder filas.

In [2]:
print("--- INFO: ORDERS ---")
orders.info()
print("\n--- INFO: PRODUCTS ---")
products.info()
print("\n--- INFO: AISLES ---")
aisles.info()
print("\n--- INFO: DEPARTMENTS ---")
departments.info()
print("\n--- INFO: ORDER_PRODUCTS ---")
# Usamos show_counts=True porque en tablas gigantes Pandas lo oculta por defecto
order_products.info(show_counts=True)

--- INFO: ORDERS ---
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB

--- INFO: PRODUCTS ---
<class 'pandas.DataFrame'>
RangeIndex: 49694 entries, 0 to 49693
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49694 non-null  int64
 1   product_name   48436 non-null  str  
 2   aisle_id       49694 non-null  int64
 3   department_id  49694 non-null  int64
dtypes: int64(3), str(1)

## Momento 3: Reconocimiento de la estructura y calidad

### Tabla de diagnóstico de los datos

| Archivo | Cantidad de filas | Cantidad de columnas | Observaciones sobre los datos (nulos) |
| :--- | :--- | :--- | :--- |
| **orders** | 478.967 | 6 | Tiene valores nulos en `days_since_prior_order` (32.876 faltantes). Esto tiene sentido lógico, ya que la primera compra de un usuario no tiene un pedido previo. |
| **products** | 49.694 | 4 | Tiene valores nulos en `product_name` (1.258 faltantes). Hay productos sin nombre asignado. |
| **aisles** | 134 | 2 | No presenta valores nulos. Datos completos. |
| **departments** | 21 | 2 | No presenta valores nulos. Datos completos. |
| **order_products** | 4.545.007 | 4 | Tiene valores nulos en `add_to_cart_order` (836 faltantes). Algunas compras no registraron el orden de ingreso al carrito. |

*Nota conceptual:* Estos conjuntos se relacionan entre sí a través de sus llaves primarias y foráneas (`order_id`, `product_id`, `aisle_id` y `department_id`).

### Respuestas de análisis (Momento 3)

- **¿Qué diferencias existen entre int64, float64 y object?**
  - `int64`: Almacena números enteros sin decimales (ej. IDs, cantidad de productos).
  - `float64`: Almacena números con decimales. Además, en Pandas, si una columna de enteros tiene algún valor nulo (NaN), automáticamente se convierte a float64.
  - `object`: Almacena cadenas de texto o datos mixtos (ej. nombres de productos).
  
- **¿Qué información proporciona info() que no se obtiene directamente con shape?**
  `shape` solo entrega las dimensiones brutas (filas y columnas). `info()` va más allá: detalla el nombre de cada columna, su tipo de dato, la cantidad exacta de valores no nulos y el peso total del archivo en la memoria RAM.
  
- **¿En qué columnas se observan valores ausentes?**
  En `days_since_prior_order` (de orders), `product_name` (de products) y `add_to_cart_order` (de order_products).
  
- **¿Qué posibles dificultades podrían generar estos valores en un análisis posterior?**
  Los datos nulos pueden causar errores matemáticos al intentar calcular promedios, alterar visualizaciones gráficas, o generar errores en modelos predictivos si no se limpian o rellenan correctamente antes de operar con ellos.

## Momento 4: Indexación y filtrado lógico de la información
A continuación, ejecutaremos 10 consultas (8 simples y 2 compuestas) utilizando el método `.loc[]` de Pandas para filtrar filas y seleccionar columnas.

In [3]:
# 1. Seleccione únicamente product_id y product_name
ej1 = products.loc[:, ["product_id", "product_name"]]
print("1. Solo product_id y product_name:", ej1.shape[0], "filas. (Solo extrajimos 2 columnas verticales).")

# 2. Consulte los productos cuyo department_id sea igual a 1
ej2 = products.loc[products["department_id"] == 1]
print("2. Productos del departamento 1:", ej2.shape[0], "productos pertenecen a este departamento.")

# 3. Seleccione los pedidos realizados a las 10 horas
ej3 = orders.loc[orders["order_hour_of_day"] == 10]
print("3. Pedidos a las 10 horas:", ej3.shape[0], "pedidos realizados en ese horario.")

# 4. Filtre los pedidos cuyo order_dow sea igual a 0
ej4 = orders.loc[orders["order_dow"] == 0]
print("4. Pedidos en el día 0 (dow=0):", ej4.shape[0], "pedidos realizados en el día cero de la semana.")

# 5. Consulte los pedidos con order_number superior a 10
ej5 = orders.loc[orders["order_number"] > 10]
print("5. Pedidos con número superior a 10:", ej5.shape[0], "pedidos corresponden a clientes frecuentes (>10).")

# 6. Pedidos cuyo days_since_prior_order sea superior a 15
ej6 = orders.loc[orders["days_since_prior_order"] > 15]
print("6. Más de 15 días desde la última compra:", ej6.shape[0], "pedidos tardaron más de dos semanas.")

# 7. Seleccione registros cuyo valor de reordered sea igual a 1
ej7 = order_products.loc[order_products["reordered"] == 1]
print("7. Productos reordenados:", ej7.shape[0], "productos fueron vueltos a comprar (recompras).")

# 8. Productos en la primera posición del carrito
ej8 = order_products.loc[order_products["add_to_cart_order"] == 1]
print("8. Primer artículo al carrito:", ej8.shape[0], "productos fueron los primeros en ser agregados.")

# 9. FILTRO DOBLE: Pedidos a las 10 horas Y con order_number superior a 10
ej9 = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
print("9. Condición Doble (Pedidos):", ej9.shape[0], "pedidos cumplen ambas condiciones simultáneamente.")

# 10. FILTRO DOBLE: Departamento 1 Y aisle_id superior a 50
ej10 = products.loc[(products["department_id"] == 1) & (products["aisle_id"] > 50)]
print("10. Condición Doble (Productos):", ej10.shape[0], "productos cumplen ambas clasificaciones.")

1. Solo product_id y product_name: 49694 filas. (Solo extrajimos 2 columnas verticales).
2. Productos del departamento 1: 4008 productos pertenecen a este departamento.
3. Pedidos a las 10 horas: 40578 pedidos realizados en ese horario.
4. Pedidos en el día 0 (dow=0): 84090 pedidos realizados en el día cero de la semana.
5. Pedidos con número superior a 10: 244072 pedidos corresponden a clientes frecuentes (>10).
6. Más de 15 días desde la última compra: 110706 pedidos tardaron más de dos semanas.
7. Productos reordenados: 2683838 productos fueron vueltos a comprar (recompras).
8. Primer artículo al carrito: 450046 productos fueron los primeros en ser agregados.
9. Condición Doble (Pedidos): 21537 pedidos cumplen ambas condiciones simultáneamente.
10. Condición Doble (Productos): 1618 productos cumplen ambas clasificaciones.


### Respuestas de análisis (Momento 4)

- **¿Qué diferencia existe entre seleccionar columnas y filtrar filas?**
  Seleccionar columnas extrae únicamente ciertas variables de interés para *toda* la base de datos (es un corte vertical). Filtrar filas restringe qué registros enteros se muestran basados en una condición lógica (es un corte horizontal).
  
- **¿Qué ventajas ofrece loc[] al consultar información específica?**
  Permite aplicar un filtro lógico en las filas y, al mismo tiempo, especificar por su nombre exacto qué columnas queremos ver, todo en una sola línea de código muy legible.
  
- **¿Qué ocurre con la cantidad de registros cuando se aplican varias condiciones?**
  Usualmente, la cantidad disminuye. Al encadenar condiciones con el operador "Y" (`&`), un registro debe cumplir *todas* las reglas al mismo tiempo para no ser descartado, haciendo el filtro mucho más estricto.
  
- **¿Por qué es importante utilizar correctamente los operadores lógicos?**
  Porque un error en el operador (como usar "O" en lugar de "Y") cambiaría por completo la pregunta que le hacemos a la base de datos, arrojando resultados erróneos sin que Python muestre un fallo técnico, lo que llevaría a tomar malas decisiones en el negocio.

## Momento 5: Obtención e interpretación de indicadores básicos
En esta sección calcularemos métricas estadísticas (conteos, promedios, sumas y descripciones generales) para entender el volumen de la operación y la frecuencia de compra de los usuarios.

In [4]:
# 1. ¿Cuántos registros de pedidos contiene el archivo?
total_orders = orders.shape[0]
print(f"1. Total registros de pedidos: {total_orders}")

# 2. ¿Cuántos registros de productos contiene el catálogo?
total_products = products.shape[0]
print(f"2. Total registros de productos en catálogo: {total_products}")

# 3. ¿Cuál es el promedio de order_number?
promedio_order_number = orders["order_number"].mean()
print(f"3. Promedio de número de pedido (order_number): {promedio_order_number:.2f}")

# 4. ¿Cuál es el promedio de days_since_prior_order?
promedio_dias = orders["days_since_prior_order"].mean()
print(f"4. Promedio de días desde pedido anterior: {promedio_dias:.2f}")

# 5. ¿Cuántos registros de artículos tienen reordered igual a 1?
conteo_recompras = order_products.loc[order_products["reordered"] == 1, "reordered"].count()
print(f"5. Artículos recomprados (reordered=1): {conteo_recompras}")

# 6. ¿Cuál es la suma de la columna reordered?
suma_recompras = order_products["reordered"].sum()
print(f"6. Suma de la columna reordered: {suma_recompras}")

# 7. Características estadísticas de order_hour_of_day
print("\n7. Estadísticas de order_hour_of_day:")
print(orders["order_hour_of_day"].describe())

# 8. Características estadísticas de add_to_cart_order
print("\n8. Estadísticas de add_to_cart_order:")
print(order_products["add_to_cart_order"].describe())

1. Total registros de pedidos: 478967
2. Total registros de productos en catálogo: 49694
3. Promedio de número de pedido (order_number): 17.16
4. Promedio de días desde pedido anterior: 11.10
5. Artículos recomprados (reordered=1): 2683838
6. Suma de la columna reordered: 2683838

7. Estadísticas de order_hour_of_day:
count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000
Name: order_hour_of_day, dtype: float64

8. Estadísticas de add_to_cart_order:
count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01
Name: add_to_cart_order, dtype: float64


### Tabla de Resultados e Interpretación

| Pregunta | Operación | Interpretación del valor obtenido |
| :--- | :--- | :--- |
| **Total pedidos** | `shape` | El archivo contiene 478.967 registros de facturas. **Ojo:** Esto no significa que haya 478 mil clientes diferentes, ya que un mismo usuario (`user_id`) puede tener múltiples pedidos. |
| **Total productos** | `shape` | Instacart tiene un catálogo masivo de 49.694 artículos diferentes disponibles. |
| **Promedio order_number** | `mean()` | Indica que el ticket promedio registrado en la base de datos suele ser la 17ª compra del usuario, demostrando alta fidelidad. |
| **Promedio días previos** | `mean()` | Los clientes tardan en promedio un poco más de 11 días en volver a surtir su mercado en la plataforma. |
| **Conteo recompras (reordered=1)** | `count()` | Existen más de 2.6 millones de registros dentro de los carritos que corresponden a productos que el cliente ya había comprado en el pasado. |
| **Suma de reordered** | `sum()` | Da exactamente el mismo número que el conteo anterior. Al ser una columna binaria (0 o 1), sumar todos los 1s equivale matemáticamente a contarlos. |
| **Stats hora de compra** | `describe()` | Las compras ocurren las 24 horas (min: 0, max: 23). Sin embargo, el percentil 50 indica que la mitad de los pedidos suceden antes de las 14:00 (2:00 PM), siendo el pico al mediodía. |
| **Stats artículos por carrito** | `describe()` | El usuario promedio añade entre 8 y 9 productos a su carrito por pedido. El carrito más lleno registrado tuvo 80 artículos. |

## Momento 6: Resolución del problema y síntesis final

**1. Respuestas a las 3 preguntas de análisis iniciales:**
*   **¿A qué hora del día se realiza la mayor cantidad de pedidos?** Según las estadísticas descriptivas obtenidas, la mitad de los pedidos ocurren antes de las 14:00 horas, concentrándose fuertemente alrededor del mediodía.
*   **¿Qué proporción de productos comprados suelen ser recompras?** Se registró que más de 2.6 millones de artículos añadidos al carrito corresponden a productos previamente comprados (reordered = 1), lo que representa más del 50% de los 4.5 millones de registros en los carritos.
*   **¿Cuántos días en promedio tardan los clientes en volver a pedir?** Los usuarios esperan en promedio 11,11 días entre un pedido y el siguiente.

**2. Principales hallazgos sobre pedidos y productos:**
Instacart maneja un volumen transaccional masivo. Cuenta con un catálogo muy amplio de casi 50.000 productos. El comportamiento general revela que el cliente promedio no hace compras de "despensa mensual gigante", sino compras recurrentes quincenales o semanales, llevando entre 8 y 9 productos por carrito.

**3. La utilidad de la indexación y los filtros:**
El uso de `.loc[]` y los filtros booleanos fue fundamental para explorar los datos sin saturar el sistema. Al tener un archivo de más de 4.5 millones de registros, intentar revisarlo manualmente o en Excel sería imposible. Los filtros permitieron aislar rápidamente reglas de negocio específicas (por ejemplo, cruzar el departamento de un producto con su pasillo) en fracciones de segundo.

**4. Limitaciones identificadas en los datos:**
*   **Valores Nulos:** Se identificó la falta de nombres en algunos productos (`product_name`), lo que denota problemas de calidad en el catálogo. Por otro lado, los nulos en `days_since_prior_order` son lógicos (representan la primera compra de un cliente nuevo), pero requieren tratamiento si se van a entrenar modelos matemáticos.
*   **Ambigüedad:** La columna `order_dow` (día de la semana) va de 0 a 6, pero el diccionario de datos no aclara si el 0 representa el domingo o el lunes, lo que dificulta un análisis preciso sobre fines de semana.

**5. Conclusiones finales:**
1. **Alta fidelidad de los clientes:** La plataforma tiene una fuerte retención. El hecho de que el promedio de número de pedidos sea superior a 17 y que más de la mitad de la canasta sean recompras demuestra un hábito de consumo muy arraigado.
2. **Logística y horarios:** Los picos de tráfico ocurren al mediodía. Esta información es crucial para asignar más servidores a la app y asegurar mayor disponibilidad de repartidores en esa franja.
3. **Estrategia de interfaz:** Dado que la mayoría de los productos son recompras, la aplicación debe destacar un botón de "Volver a comprar mi última canasta" apenas el usuario inicia sesión, reduciendo la fricción y asegurando la venta rápida.